In [1]:
!pip -q install datasets transformers sentence-transformers faiss-cpu rank-bm25 evaluate accelerate scikit-learn gensim nltk gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 73.2 MB/s eta 0:00:00:00:0100:01


In [ ]:
import os
import re
import gc
import json
import time
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder

from datasets import load_dataset, Dataset
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    pipeline
)

from sentence_transformers import SentenceTransformer, CrossEncoder

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss
import gradio as gr

from gensim.models import Word2Vec, FastText

warnings.filterwarnings('ignore')

In [3]:
for pkg in ["punkt", "stopwords", "wordnet", "omw-1.4"]:
    try:
        nltk.download(pkg, quiet=True)
    except:
        pass

## 1. Load Dataset

In [4]:
dataset = load_dataset(
    "PolyAI/banking77",
    revision="830c4f2be6949546b11fe83fbc50993348a2bccd"
)
dataset

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/295k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/93.0k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3080 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})

In [5]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path("./finassist_project")
PROJECT_DIR.mkdir(exist_ok=True, parents=True)
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True, parents=True)
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True, parents=True)
REPORT_DIR = PROJECT_DIR / "reports"
REPORT_DIR.mkdir(exist_ok=True, parents=True)

In [6]:
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

label_names = dataset["train"].features["label"].names
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in id2label.items()}

train_df["label_name"] = train_df["label"].map(id2label)
test_df["label_name"] = test_df["label"].map(id2label)

In [7]:
print(train_df.shape, test_df.shape)
train_df.head()

(10003, 3) (3080, 3)


,text,label,label_name
0,I am still waiting on my card?,11,card_arrival
1,What can I do if my card still hasn't arrived ...,11,card_arrival
2,I have been waiting over a week. Is the card s...,11,card_arrival
3,Can I track my card while it is in the process...,11,card_arrival
4,"How do I know if I will get my card, or if it ...",11,card_arrival


In [8]:
train_df["text_len_words"] = train_df["text"].str.split().str.len()
train_df["text_len_chars"] = train_df["text"].str.len()

display(train_df[["text_len_words", "text_len_chars"]].describe())
display(train_df["label_name"].value_counts().head(15))

,text_len_words,text_len_chars
count,10003.000000,10003.000000
mean,11.949415,59.473758
std,7.891577,40.867901
min,2.000000,13.000000
25%,7.000000,36.000000
50%,10.000000,47.000000
75%,13.000000,64.000000
max,79.000000,433.000000


,count
label_name,
card_payment_fee_charged,187
direct_debit_payment_not_recognised,182
balance_not_updated_after_cheque_or_cash_deposit,181
wrong_amount_of_cash_received,180
cash_withdrawal_charge,177
transaction_charged_twice,175
declined_cash_withdrawal,173
transfer_fee_charged,172
balance_not_updated_after_bank_transfer,171


## 2. Text preprocessing


In [9]:
STOPWORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [10]:
def basic_clean(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"@[A-Za-z0-9_]+", " <MENTION> ", text)
    text = re.sub(r"#[A-Za-z0-9_]+", " <HASHTAG> ", text)
    text = re.sub(r"\d+", " <NUMBER> ", text)
    text = re.sub(r"[^a-z<>\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def classical_preprocess(text : str) ->str:
    text = basic_clean(text)
    tokens = []
    for tok in text.split():
        if tok in STOPWORDS:
            continue
        if tok.startswith("<") and tok.endswith(">"):
            tokens.append(tok)
        else:
            tokens.append(lemmatizer.lemmatize(tok))
    return " ".join(tokens)

In [11]:
train_df["text_clean"] = train_df["text"].apply(classical_preprocess)
test_df["text_clean"] = test_df["text"].apply(classical_preprocess)

train_df[["text", "text_clean"]].head(10)

,text,text_clean
0,I am still waiting on my card?,still waiting card
1,What can I do if my card still hasn't arrived ...,card still arrived < > week
2,I have been waiting over a week. Is the card s...,waiting week card still coming
3,Can I track my card while it is in the process...,track card process delivery
4,"How do I know if I will get my card, or if it ...",know get card lost
5,When did you send me my new card?,send new card
6,Do you have info about the card on delivery?,info card delivery
7,What do I do if I still have not received my n...,still received new card
8,Does the package with my card have tracking?,package card tracking
9,I ordered my card but it still isn't here,ordered card still


## 3. Classical baseline: TF-IDF + Logistic Regression


In [12]:
tfidf_clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        max_features=50000
    )),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

tfidf_clf.fit(train_df["text_clean"], train_df["label_name"])
tfidf_preds = tfidf_clf.predict(test_df["text_clean"])

tfidf_acc = accuracy_score(test_df["label_name"], tfidf_preds)
tfidf_f1 = f1_score(test_df["label_name"], tfidf_preds, average="weighted")

print("TF-IDF Accuracy:", round(tfidf_acc, 4))
print("TF-IDF Weighted F1:", round(tfidf_f1, 4))

TF-IDF Accuracy: 0.8523
TF-IDF Weighted F1: 0.8526


## 4. Dense embeddings: Word2Vec and FastText

In [20]:
tokenized_sentences = [x.split() for x in train_df["text_clean"].tolist()]

w2v_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    seed=SEED
)

ft_model = FastText(
    sentences=tokenized_sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=SEED
)

print("FastText OOV-style vector:", ft_model.wv["virtualcard"][:5])

FastText OOV-style vector: [-0.19727418 -0.17898983  0.12934777 -0.02321996 -0.05838147]


In [21]:
def average_embedding(text, model, dim=100):
    tokens = text.split()
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    if len(vecs) == 0:
        return np.zeros(dim, dtype=np.float32)
    
    return np.mean(vecs, axis=0).astype(np.float32)

x_train_w2v = np.stack([average_embedding(t, w2v_model, 100) for t in train_df["text_clean"]])
x_test_w2v = np.stack([average_embedding(t, w2v_model, 100) for t in test_df["text_clean"]])

In [22]:
w2v_lr = LogisticRegression(max_iter=2000, class_weight='balanced')
w2v_lr.fit(x_train_w2v, train_df["label_name"])

w2v_preds = w2v_lr.predict(x_test_w2v)

w2v_acc = accuracy_score(test_df["label_name"], w2v_preds)
w2v_f1 = f1_score(test_df["label_name"], w2v_preds, average='weighted')

print("Word2Vec AvgEmb Accuracy:", round(w2v_acc, 4))
print("Word2Vec AvgEmb Weighted F1:", round(w2v_f1, 4))

Word2Vec AvgEmb Accuracy: 0.5838
Word2Vec AvgEmb Weighted F1: 0.5731


## 5. BiLSTM baseline

In [13]:
MAX_VOCAB = 20000
MAX_LEN = 40
EMBED_DIM = 100
HIDDEN_DIM = 128
BATCH_SIZE = 128
EPOCHS_LSTM = 8
DEVICE = "cuda"

In [14]:
label_encoder = LabelEncoder()
word_counter = Counter()

y_train_enc = label_encoder.fit_transform(train_df["label_name"])
y_test_enc = label_encoder.transform(test_df["label_name"])

In [15]:
for text in train_df["text_clean"]:
    word_counter.update(text.split())
    
special_tokens = ["<PAD>", "<UNK>"]
vocab_words = [w for w, _ in word_counter.most_common(MAX_VOCAB - len(special_tokens))]
vocab = {w: i for i, w in enumerate(special_tokens + vocab_words)}

pad_id = vocab["<PAD>"]
unk_id = vocab["<UNK>"]

In [16]:
def encode_text(text, max_len=MAX_LEN):
    ids = [vocab.get(tok, unk_id) for tok in text.split()][:max_len]
    if len(ids) < max_len:
        ids += [pad_id] * (max_len - len(ids))
    return ids

x_train_ids = np.array([encode_text(t) for t in train_df["text_clean"]], dtype=np.int64)
x_test_ids = np.array([encode_text(t) for t in test_df["text_clean"]], dtype=np.int64)

In [17]:
class TextDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
        
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

train_loader = DataLoader(TextDataset(x_train_ids, y_train_enc), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TextDataset(x_test_ids, y_test_enc), batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
embedding_matrix = np.random.normal(0, 0.1, size=(len(vocab), EMBED_DIM)).astype(np.float32)
embedding_matrix[pad_id] = 0